In [1]:
# Imports
import pandas as pd
import numpy as np

In [2]:
# Load SPECTER embeddings
spe_embd = pd.read_csv(r"data/specter_embeddings.csv").drop(columns=['original_id_clean'])  

In [3]:
# Load GraphSAGE embeddings
gs_embd = pd.read_csv(r"kg/project_graphsage_embeddings.csv")

In [4]:
# from sklearn.metrics.pairwise import cosine_similarity

# # IDs
# spec_ids = spe_embd.iloc[:,0].astype(int).to_numpy()
# gs_ids   = gs_embd.iloc[:,0].astype(int).to_numpy()

# # Label lookup
# y = pd.read_csv(r"data/targets.csv")
# y_map = dict(zip(y["project_id"].astype(int), y["is_highly_cited"].astype(int)))

# # Keep only rows with labels and common IDs
# common = sorted(set(spec_ids) & set(gs_ids) & set(y_map.keys()))
# spec_idx = {pid:i for i,pid in enumerate(spec_ids)}
# gs_idx   = {pid:i for i,pid in enumerate(gs_ids)}

# Xspec = l2_normalize(spe_embd.iloc[[spec_idx[p] for p in common], 1:].to_numpy())
# Xgs   = l2_normalize(gs_embd.iloc[[gs_idx[p]   for p in common], 1:].to_numpy())
# yvec  = np.array([y_map[p] for p in common])

# def knn_homophily(X, y, k=20, batch=1024):
#     n = X.shape[0]
#     same = []
#     for start in range(0, n, batch):
#         end = min(n, start+batch)
#         S = X[start:end] @ X.T              # cosine because normalized
#         np.fill_diagonal(S[:, start:end], -np.inf)  # remove self within block
#         nn = np.argpartition(-S, kth=k, axis=1)[:, :k]
#         same.append((y[nn] == y[start:end, None]).mean(axis=1))
#     return float(np.mean(np.concatenate(same)))

# print("Homophily@20 SPECTER:", knn_homophily(Xspec, yvec, k=20))
# print("Homophily@20 GraphSAGE:", knn_homophily(Xgs, yvec, k=20))

In [5]:
# p = yvec.mean()
# baseline = p**2 + (1-p)**2
# print("label prevalence p:", float(p))
# print("random-match baseline p^2+(1-p)^2:", float(baseline))

In [6]:
# Embedding matrices (drop the id column)
gs_embd_np = gs_embd.iloc[:,1:].to_numpy()
spe_embd_np = spe_embd.iloc[:,1:].to_numpy()

In [7]:
# Shapes
spe_embd_np.shape, gs_embd_np.shape

((131772, 768), (131772, 768))

In [8]:
# NaN check
np.isnan(spe_embd_np).sum(), np.isnan(gs_embd_np).sum()

(np.int64(0), np.int64(0))

In [9]:
# L2-normalize both so dot product = cosine
from sklearn.preprocessing import normalize
specter_norm = normalize(spe_embd_np, norm='l2', axis=1)
graphsage_norm = normalize(gs_embd_np, norm='l2', axis=1)

In [10]:
# Verify unit norm
print(np.mean(np.linalg.norm(specter_norm, axis=1)))
print(np.mean(np.linalg.norm(graphsage_norm, axis=1)))

1.0
1.0


In [11]:
# Mean cosine between random pairs (anisotropy check)
import random
def rand_cosine_mean(emb,n=10000):
    sims = []
    N = emb.shape[0]
    for _ in range(n):
        i,j = random.randint(0,N-1), random.randint(0,N-1)
        sims.append(emb[i].dot(emb[j]))
    return np.mean(sims)
print("Random pairwise cosine mean Specter:", rand_cosine_mean(specter_norm))
print("Random pairwise cosine mean GraphSAGE:", rand_cosine_mean(graphsage_norm))

Random pairwise cosine mean Specter: 0.6771139709914994
Random pairwise cosine mean GraphSAGE: 0.23154060294589218


In [12]:
# Mean per-dimension variance
np.mean(np.var(gs_embd_np, axis=0)), np.mean(np.var(spe_embd_np, axis=0))

(np.float64(3774.624093212293), np.float64(0.21067738098347263))

In [13]:
# PCA explained variance of GraphSAGE (embedding-collapse check)
from sklearn.decomposition import PCA
pca = PCA(n_components=10)
pca.fit(gs_embd_np)
pca.explained_variance_ratio_

array([8.97233337e-01, 6.39184975e-02, 2.12527886e-02, 1.06035452e-02,
       3.19612104e-03, 2.22922798e-03, 7.29318131e-04, 2.87961050e-04,
       1.47725837e-04, 7.09238691e-05])

In [14]:
# Random-pair cosine stats (vectorized)
import numpy as np

def rand_cosine_stats(X, n=20000, seed=0):
    rng = np.random.default_rng(seed)
    N = X.shape[0]
    i = rng.integers(0, N, size=n)
    j = rng.integers(0, N, size=n)
    mask = (i != j)
    i, j = i[mask], j[mask]

    cos = np.einsum("ij,ij->i", X[i], X[j])  # dot because rows already L2-normalized
    return {
        "N": N,
        "pairs_used": cos.size,
        "mean": float(cos.mean()),
        "std": float(cos.std()),
        "min": float(cos.min()),
        "p1,p50,p99": [float(x) for x in np.percentile(cos, [1, 50, 99])]
    }

print("SPECTER:", rand_cosine_stats(specter_norm))
print("GraphSAGE:", rand_cosine_stats(graphsage_norm))

SPECTER: {'N': 131772, 'pairs_used': 20000, 'mean': 0.6775002709481467, 'std': 0.08354524467821234, 'min': 0.3251475612451943, 'p1,p50,p99': [0.44843927916644133, 0.6870082995496767, 0.8397920892974403]}
GraphSAGE: {'N': 131772, 'pairs_used': 20000, 'mean': 0.22092369533209813, 'std': 0.5680967350948479, 'min': -0.9329110598487, 'p1,p50,p99': [-0.8650391412399867, 0.2965599283791759, 0.9892701136891427]}


In [15]:
# Embedding-collapse diagnostics: row/feature variance and unique-row count
X = graphsage_norm
print("Row variance mean:", float(np.var(X, axis=1).mean()))
print("Feature variance mean:", float(np.var(X, axis=0).mean()))

# How many unique rows (approx) — round to reduce float noise
Xu = np.unique(np.round(X, 6), axis=0)
print("Unique rows (rounded 1e-6):", Xu.shape[0], "out of", X.shape[0])

# Compare a few rows directly to row 0
print("max |x0-xi| for i=1..5:", [float(np.max(np.abs(X[0]-X[i]))) for i in range(1,6)])

Row variance mean: 0.0013017639879931706
Feature variance mean: 0.001012502889343915
Unique rows (rounded 1e-6): 131772 out of 131772
max |x0-xi| for i=1..5: [0.0730398265280404, 0.03657559107002065, 0.055172176780055954, 0.026201874129599352, 0.021963812970356125]


In [16]:
# Mean/std cosine over graph edges
def edge_smoothness(edges,emb):
    sims = []
    for i,j in edges:
        sims.append(emb[i].dot(emb[j]))
    return np.mean(sims), np.std(sims)


In [17]:
# Top-k neighbours via full cosine (small scale)
from sklearn.metrics.pairwise import cosine_similarity
def topk_neighbors(emb,idx,k=5):
    sims = cosine_similarity(emb[idx:idx+1], emb)[0]
    return np.argsort(-sims)[1:k+1]

In [18]:
# Sample 100 random anchor nodes
anchors = np.random.choice(len(gs_embd_np), size=100, replace=False)

In [19]:
# Jaccard overlap of SPECTER vs GraphSAGE top-20 neighbours
def jaccard(a,b):
    return len(set(a) & set(b)) / len(set(a) | set(b))

jaccards = []
for i in anchors:
    spec_nbrs = topk_neighbors(specter_norm, i, k=20)
    gs_nbrs   = topk_neighbors(graphsage_norm, i, k=20)
    jaccards.append(jaccard(spec_nbrs, gs_nbrs))
print("Mean Jaccard similarity of top-20 neighbors over 100 anchors:", np.mean(jaccards)    )

Mean Jaccard similarity of top-20 neighbors over 100 anchors: 0.03858501542712069


In [23]:
# Jaccard overlap: observed vs random baselines
k = 20
N = specter_norm.shape[0]
assert graphsage_norm.shape[0] == N, "specter_norm and graphsage_norm must align by row/id"

rng = np.random.default_rng(0)

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b)

def random_neighbors(idx, N, k, rng):
    # sample k nodes uniformly at random, excluding idx
    candidates = np.delete(np.arange(N), idx)
    return rng.choice(candidates, size=k, replace=False)

jacc_obs = []
jacc_spec_vs_rand = []
jacc_gs_vs_rand = []
jacc_rand_vs_rand = []

for i in anchors:
    spec_nbrs = topk_neighbors(specter_norm, i, k=k)
    gs_nbrs   = topk_neighbors(graphsage_norm, i, k=k)

    rand1 = random_neighbors(i, N, k, rng)
    rand2 = random_neighbors(i, N, k, rng)

    jacc_obs.append(jaccard(spec_nbrs, gs_nbrs))
    jacc_spec_vs_rand.append(jaccard(spec_nbrs, rand1))   # naive SPECTER baseline
    jacc_gs_vs_rand.append(jaccard(gs_nbrs, rand1))       # naive GraphSAGE baseline
    jacc_rand_vs_rand.append(jaccard(rand1, rand2))       # pure chance baseline

print(f"Observed mean Jaccard@{k} (SPECTER vs GraphSAGE):", float(np.mean(jacc_obs)))
print(f"Naive mean Jaccard@{k} (SPECTER vs Random):      ", float(np.mean(jacc_spec_vs_rand)))
print(f"Naive mean Jaccard@{k} (GraphSAGE vs Random):     ", float(np.mean(jacc_gs_vs_rand)))
print(f"Chance mean Jaccard@{k} (Random vs Random):       ", float(np.mean(jacc_rand_vs_rand)))

# Optional: theoretical expectation for two random k-sets from (N-1) candidates
Nprime = N - 1
expected_random_jacc = k / (2 * Nprime - k)
print(f"Theoretical E[Jaccard] for random k-sets (N={N}): ", float(expected_random_jacc))

Observed mean Jaccard@20 (SPECTER vs GraphSAGE): 0.03858501542712069
Naive mean Jaccard@20 (SPECTER vs Random):       0.0
Naive mean Jaccard@20 (GraphSAGE vs Random):      0.0
Chance mean Jaccard@20 (Random vs Random):        0.0
Theoretical E[Jaccard] for random k-sets (N=131772):  7.589499168949841e-05


In [20]:
# Recall@20 of GraphSAGE neighbours against SPECTER neighbours
def recall_at_k(a,b):
    return len(set(a) & set(b)) / len(set(a))
recalls = []
for i in anchors:
    spec_nbrs = topk_neighbors(specter_norm, i, k=20)
    gs_nbrs   = topk_neighbors(graphsage_norm, i, k=20)
    recalls.append(recall_at_k(gs_nbrs, spec_nbrs))
print("Mean Recall@20 over 100 anchors:", np.mean(recalls))

Mean Recall@20 over 100 anchors: 0.0715


In [24]:
# Recall@20: observed vs random baselines
k = 20
N = specter_norm.shape[0]
assert graphsage_norm.shape[0] == N, "specter_norm and graphsage_norm must align by row/id"

rng = np.random.default_rng(0)

def recall_at_k(a, b):
    # fraction of items in 'a' that also appear in 'b'
    a, b = set(a), set(b)
    return len(a & b) / len(a)

def random_neighbors(idx, N, k, rng):
    candidates = np.delete(np.arange(N), idx)   # exclude self
    return rng.choice(candidates, size=k, replace=False)

obs = []
gs_vs_rand = []
rand_vs_spec = []
rand_vs_rand = []

for i in anchors:
    spec_nbrs = topk_neighbors(specter_norm, i, k=k)
    gs_nbrs   = topk_neighbors(graphsage_norm, i, k=k)

    rand1 = random_neighbors(i, N, k, rng)
    rand2 = random_neighbors(i, N, k, rng)

    obs.append(recall_at_k(gs_nbrs, spec_nbrs))        # your original metric
    gs_vs_rand.append(recall_at_k(gs_nbrs, rand1))     # naive GraphSAGE baseline
    rand_vs_spec.append(recall_at_k(rand1, spec_nbrs)) # naive SPECTER baseline (random retrieved vs SPECTER set)
    rand_vs_rand.append(recall_at_k(rand1, rand2))     # pure chance baseline

print(f"Observed mean Overlap@{k} (GraphSAGE in SPECTER):", float(np.mean(obs)))
print(f"Naive mean Overlap@{k} (GraphSAGE in Random):    ", float(np.mean(gs_vs_rand)))
print(f"Naive mean Overlap@{k} (Random in SPECTER):      ", float(np.mean(rand_vs_spec)))
print(f"Chance mean Overlap@{k} (Random in Random):      ", float(np.mean(rand_vs_rand)))

# Theoretical expectation for random k-sets from (N-1) candidates:
expected = k / (N - 1)
print(f"Theoretical E[Overlap@{k}] for random sets (N={N}):", float(expected))

Observed mean Overlap@20 (GraphSAGE in SPECTER): 0.0715
Naive mean Overlap@20 (GraphSAGE in Random):     0.0
Naive mean Overlap@20 (Random in SPECTER):       0.0
Chance mean Overlap@20 (Random in Random):       0.0
Theoretical E[Overlap@20] for random sets (N=131772): 0.00015177846415372122


In [21]:
# Spearman correlation of neighbour similarities (top-50)
from scipy.stats import spearmanr

def rank_corr(emb1, emb2, idx, k=50, eps=1e-12):
    sims1 = cosine_similarity(emb1[idx:idx+1], emb1)[0]
    sims2 = cosine_similarity(emb2[idx:idx+1], emb2)[0]

    order = np.argsort(-sims1)
    order = order[order != idx]      # drop self
    top = order[:k]

    a = sims1[top]
    b = sims2[top]

    # Avoid ConstantInputWarning / undefined correlation
    if np.std(a) < eps or np.std(b) < eps:
        return np.nan

    return spearmanr(a, b).correlation

In [22]:
# Mean neighbour-rank correlation over anchors
cors = [rank_corr(specter_norm, graphsage_norm, i, k=50) for i in anchors]
print("Mean Spearman rank correlation of top-50 neighbors over 100 anchors:", np.nanmean(cors))

Mean Spearman rank correlation of top-50 neighbors over 100 anchors: 0.21349147659063625


In [25]:
# Rank correlation with a shuffle baseline (refined version of the cell above)
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def rank_corr(emb1, emb2, idx, k=50, eps=1e-12):
    sims1 = cosine_similarity(emb1[idx:idx+1], emb1)[0]
    sims2 = cosine_similarity(emb2[idx:idx+1], emb2)[0]

    order = np.argsort(-sims1)
    order = order[order != idx]      # drop self
    top = order[:k]

    a = sims1[top]
    b = sims2[top]

    if np.std(a) < eps or np.std(b) < eps:
        return np.nan

    return spearmanr(a, b).correlation

def rank_corr_shuffle_b(emb1, emb2, idx, k=50, eps=1e-12, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    sims1 = cosine_similarity(emb1[idx:idx+1], emb1)[0]
    sims2 = cosine_similarity(emb2[idx:idx+1], emb2)[0]

    order = np.argsort(-sims1)
    order = order[order != idx]
    top = order[:k]

    a = sims1[top]
    b = sims2[top]

    if np.std(a) < eps or np.std(b) < eps:
        return np.nan

    return spearmanr(a, rng.permutation(b)).correlation  # chance baseline ~ 0

def random_unit_matrix(N, d, rng, eps=1e-12):
    X = rng.normal(size=(N, d))
    X /= (np.linalg.norm(X, axis=1, keepdims=True) + eps)
    return X

k = 50
rng = np.random.default_rng(0)
N = specter_norm.shape[0]
assert graphsage_norm.shape[0] == N

rand_for_spec = random_unit_matrix(N, specter_norm.shape[1], rng)
rand_for_gs   = random_unit_matrix(N, graphsage_norm.shape[1], rng)

# Observed (note: asymmetric depending on which embedding defines the "top-k")
cors_spec_top = [rank_corr(specter_norm, graphsage_norm, i, k=k) for i in anchors]
cors_gs_top   = [rank_corr(graphsage_norm, specter_norm, i, k=k) for i in anchors]

# Naive baselines
cors_spec_vs_rand = [rank_corr(specter_norm, rand_for_spec, i, k=k) for i in anchors]
cors_gs_vs_rand   = [rank_corr(graphsage_norm, rand_for_gs, i, k=k) for i in anchors]
cors_shuffle       = [rank_corr_shuffle_b(specter_norm, graphsage_norm, i, k=k, rng=rng) for i in anchors]

print(f"Observed mean Spearman@{k} (SPECTER-top, compare to GraphSAGE):", float(np.nanmean(cors_spec_top)))
print(f"Observed mean Spearman@{k} (GraphSAGE-top, compare to SPECTER):", float(np.nanmean(cors_gs_top)))
print(f"Naive mean Spearman@{k} (SPECTER vs Random):                   ", float(np.nanmean(cors_spec_vs_rand)))
print(f"Naive mean Spearman@{k} (GraphSAGE vs Random):                  ", float(np.nanmean(cors_gs_vs_rand)))
print(f"Chance mean Spearman@{k} (shuffle baseline):                    ", float(np.nanmean(cors_shuffle)))

Observed mean Spearman@50 (SPECTER-top, compare to GraphSAGE): 0.21349147659063625
Observed mean Spearman@50 (GraphSAGE-top, compare to SPECTER): 0.20781575665813828
Naive mean Spearman@50 (SPECTER vs Random):                    0.0034199279711884744
Naive mean Spearman@50 (GraphSAGE vs Random):                   -0.014757262905162065
Chance mean Spearman@50 (shuffle baseline):                     0.00020264105642256891


In [26]:
# Scalable top-k neighbours via a random candidate pool
import numpy as np
from scipy.stats import spearmanr
def topk_neighbors(X, anchor_idx, K=20, pool_size=10000, seed=0):
    """
    Returns indices of top-K neighbors for one anchor
    using random candidate pool (scales to huge N).
    """
    rng = np.random.default_rng(seed)
    N = X.shape[0]

    # sample candidate pool (excluding anchor)
    candidates = rng.choice(N, size=pool_size, replace=False)
    candidates = candidates[candidates != anchor_idx]

    sims = X[candidates] @ X[anchor_idx]  # cosine (X normalized)
    topk_idx = np.argsort(-sims)[:K]

    return candidates[topk_idx], sims[topk_idx]


In [27]:
# Mean Jaccard@20 across anchors (pooled sampling)
def mean_jaccard_topk(X_base, X_gs, K=20, anchors=100, pool_size=10000):
    rng = np.random.default_rng(0)
    N = X_base.shape[0]
    anchor_nodes = rng.choice(N, size=anchors, replace=False)

    jaccards = []

    for a in anchor_nodes:
        nb_base, _ = topk_neighbors(X_base, a, K, pool_size)
        nb_gs, _   = topk_neighbors(X_gs, a, K, pool_size)

        set_base = set(nb_base.tolist())
        set_gs   = set(nb_gs.tolist())

        j = len(set_base & set_gs) / len(set_base | set_gs)
        jaccards.append(j)

    return float(np.mean(jaccards))


In [28]:
# Mean Recall@20 across anchors (pooled sampling)
def mean_recall_at_k(X_base, X_gs, K=20, anchors=100, pool_size=10000):
    rng = np.random.default_rng(1)
    N = X_base.shape[0]
    anchor_nodes = rng.choice(N, size=anchors, replace=False)

    recalls = []

    for a in anchor_nodes:
        nb_base, _ = topk_neighbors(X_base, a, K, pool_size)
        nb_gs, _   = topk_neighbors(X_gs, a, K, pool_size)

        recall = len(set(nb_base) & set(nb_gs)) / K
        recalls.append(recall)

    return float(np.mean(recalls))


In [29]:
# Mean Spearman@50 across anchors (pooled sampling)
def mean_spearman_topk(X_base, X_gs, K=50, anchors=100, pool_size=15000):
    rng = np.random.default_rng(2)
    N = X_base.shape[0]
    anchor_nodes = rng.choice(N, size=anchors, replace=False)

    scores = []

    for a in anchor_nodes:
        # use same candidate pool for fair ranking
        candidates = rng.choice(N, size=pool_size, replace=False)
        candidates = candidates[candidates != a]

        sims_base = X_base[candidates] @ X_base[a]
        sims_gs   = X_gs[candidates]   @ X_gs[a]

        topk = np.argsort(-sims_base)[:K]

        rank_base = np.arange(K)
        rank_gs   = np.argsort(-sims_gs[topk])

        rho, _ = spearmanr(rank_base, rank_gs)
        if not np.isnan(rho):
            scores.append(rho)

    return float(np.mean(scores))


In [31]:
# Run the pooled neighbour-overlap metrics
print("Mean Jaccard@20:",
      mean_jaccard_topk(specter_norm, graphsage_norm))

print("Mean Recall@20:",
      mean_recall_at_k(specter_norm, graphsage_norm))

print("Mean Spearman@50:",
      mean_spearman_topk(specter_norm, graphsage_norm))


Mean Jaccard@20: 0.05630295322245786
Mean Recall@20: 0.08900000000000001
Mean Spearman@50: 0.1726319327731093


In [32]:
# Cosine stats over a sample of graph edges
import numpy as np

def edge_cosine_stats(X, edge_index, max_edges=200_000, seed=0):
    """
    X: (N, d) L2-normalized embeddings (numpy)
    edge_index: (2, E) edges
    """
    rng = np.random.default_rng(seed)

    if hasattr(edge_index, "cpu"):
        edge_index = edge_index.cpu().numpy()

    E = edge_index.shape[1]
    idx = rng.choice(E, size=min(E, max_edges), replace=False)

    u = edge_index[0, idx]
    v = edge_index[1, idx]

    cos = np.einsum("ij,ij->i", X[u], X[v])

    return {
        "edges_used": cos.size,
        "mean": float(cos.mean()),
        "std": float(cos.std()),
        "p1,p50,p99": [float(x) for x in np.percentile(cos, [1, 50, 99])]
    }


In [35]:
# Load edges + SPECTER, map project_ids to row indices
import numpy as np
edge_list = np.load(r"kg/mutual_edges@20.npy")  # (E, 2), undirected
print(edge_list.shape)
embd = pd.read_csv(r"data/specter_embeddings.csv").drop(columns=['original_id_clean'])  
proj_ids = embd["project_id"].astype(str).values
embeddings = embd.drop(columns=['project_id']).values
id2idx = {pid: idx for idx, pid in enumerate(proj_ids)}
idx2id = {i:pid for pid,i in id2idx.items()}
print(len(id2idx))
mapped_edges = []
missing=0
for a,b in edge_list:
    a=str(a)
    b=str(b)
    if a in id2idx and b in id2idx:
        mapped_edges.append([id2idx[a], id2idx[b]])
    else:
        missing+=1
mapped_edges = np.array(mapped_edges)
print(mapped_edges.shape, missing)

(516583, 2)


131772
(516583, 2) 0


In [36]:
# Build an undirected edge_index tensor

import torch


edge_index=torch.tensor(mapped_edges, dtype=torch.long).t()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)  # Make it undirected
edge_index.shape

torch.Size([2, 1033166])

In [37]:
# Edge cosine: SPECTER vs GraphSAGE
print("SPECTER edge cosine:", edge_cosine_stats(specter_norm, edge_index))
print("GraphSAGE edge cosine:", edge_cosine_stats(graphsage_norm, edge_index))


SPECTER edge cosine: {'edges_used': 200000, 'mean': 0.9118240786057348, 'std': 0.023123903743265072, 'p1,p50,p99': [0.844650241722838, 0.9139419145762147, 0.9581965051011747]}
GraphSAGE edge cosine: {'edges_used': 200000, 'mean': 0.9500745238633844, 'std': 0.10138603564549761, 'p1,p50,p99': [0.4765738174850989, 0.9879652650044646, 0.9999768576274886]}


In [38]:
# Edge-vs-random cosine gap
def edge_vs_random_gap(X, edge_index, n_random=200_000, seed=0):
    rng = np.random.default_rng(seed)
    N = X.shape[0]

    if hasattr(edge_index, "cpu"):
        edge_index = edge_index.cpu().numpy()

    # Edge cosine
    idx = rng.choice(edge_index.shape[1], size=min(edge_index.shape[1], n_random), replace=False)
    u_e = edge_index[0, idx]
    v_e = edge_index[1, idx]
    edge_cos = np.einsum("ij,ij->i", X[u_e], X[v_e])

    # Random cosine
    u_r = rng.integers(0, N, size=edge_cos.size)
    v_r = rng.integers(0, N, size=edge_cos.size)
    mask = u_r != v_r
    rand_cos = np.einsum("ij,ij->i", X[u_r[mask]], X[v_r[mask]])

    return {
        "edge_mean": float(edge_cos.mean()),
        "random_mean": float(rand_cos.mean()),
        "gap": float(edge_cos.mean() - rand_cos.mean())
    }


In [39]:
# Run the edge-vs-random gap for both embeddings
print("SPECTER gap:", edge_vs_random_gap(specter_norm, edge_index))
print("GraphSAGE gap:", edge_vs_random_gap(graphsage_norm, edge_index))


SPECTER gap: {'edge_mean': 0.9118240786057348, 'random_mean': 0.6770773709233826, 'gap': 0.23474670768235217}
GraphSAGE gap: {'edge_mean': 0.9500745238633844, 'random_mean': 0.22277620475446977, 'gap': 0.7272983191089146}
